# Barrido de K con Greedy

Analiza el barrido K ∈ {5, 10, 15, 20, 25, 30} × 3 semillas del baseline Greedy y grafica todas las metricas que le aplican. Es el ensayo de como se van a ver los resultados de la tesis: cuando existan las corridas de FedAvg, DTE y gossip, basta agregarlas a `METHODS_TO_SHOW` y cada figura y tabla las incluye.

**Datos.** En Hypatia, desde la raiz del repo: `make run-greedy` (18 trabajos de CPU). Cuando terminen (`make sq` vacio), en la maquina local: `make pull-runs`. Los CSV y JSON quedan en `v2v/runs/k<K>/greedy_seed<semilla>.{csv,json}`.

**Como se resume cada numero** (las mismas reglas que el resto de la tesis):
- valor de una corrida = promedio del ultimo 10 % de sus episodios;
- puntos y barras = media ± 1 desviacion entre las 3 semillas;
- las metricas por par (outage, delay, potencia, vecinos) se promedian primero sobre los K agentes.

Greedy no aprende ni comparte modelos: no tiene perdida, dispersion de pesos, comunicacion ni episodios hasta converger. Esas secciones quedan listas para los otros metodos y aqui solo lo explican.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")  # v2v/, para importar analysis.py
from analysis import (
    convergence_episode,
    final_values,
    load_metadata,
    load_runs,
    per_episode_means,
    summarize_by_k,
)

RUNS_ROOT = Path(os.environ.get("V2V_RUNS_ROOT", "../runs"))
FIG_DIR = Path(os.environ.get("V2V_FIG_DIR", "../runs/figures"))
FIG_DIR.mkdir(parents=True, exist_ok=True)

METHODS_TO_SHOW = ["greedy"]  # agrega "fedavg", "dte", "gossip" cuando existan sus corridas
EXPECTED_KS = [5, 10, 15, 20, 25, 30]
EXPECTED_SEEDS = [0, 1, 2]
WINDOW = 50  # episodios de la media movil
FINAL_FRACTION = 0.1  # tramo final: el ultimo 10 % de los episodios

# Mismo color por metodo que en results.ipynb. Greedy va punteado porque no aprende.
METHODS = {
    "fedavg": {"label": "FedAvg", "color": "#2a78d6", "linestyle": "-", "marker": "o"},
    "dte": {"label": "DTE (sin federar)", "color": "#eb6834", "linestyle": "-", "marker": "s"},
    "greedy": {"label": "Greedy", "color": "#1baf7a", "linestyle": "--", "marker": "^"},
    "gossip": {"label": "Gossip", "color": "#eda100", "linestyle": "-", "marker": "D"},
}
LABELS = {
    "reward": "Recompensa por episodio",
    "cue_rate_mbps": "Tasa CUE (Mbps)",
    "v2v_delay_ms": "Delay V2V (ms)",
    "outage_rate": "Outage (%)",
    "co_channel_rate": "Canal compartido (%)",
    "channels_used": "Canales en uso",
    "mean_power_ratio": "Potencia media / Pmax",
    "max_power_rate": "Slots a Pmax (%)",
    "neighbors": "Vecinos a 300 m o menos",
    "isolated_rate": "Pares sin vecinos (%)",
    "train_loss": "Perdida D3QN",
    "weight_dispersion": "Dispersion de pesos",
    "comm_models": "Modelos transmitidos por episodio",
    "comm_mb": "MB transmitidos por episodio",
    "comm_max_load": "Carga del nodo mas cargado",
}
PERCENT = ["outage_rate", "co_channel_rate", "max_power_rate", "isolated_rate"]

# Tasa CUE de Li et al. (2022), Fig. 3, leida de la figura (aproximada). Solo para comparar tendencias.
LI_FIG3_MBPS = {
    "greedy": {5: 5.75, 10: 5.25, 15: 4.75, 20: 4.0, 25: 3.0, 30: 1.75},
    "fedavg": {5: 7.95, 10: 7.45, 15: 6.95, 20: 6.45, 25: 5.7, 30: 4.7},
}

TEXT, TEXT_2, GRID, REFERENCE = "#0b0b0b", "#52514e", "#e6e6e3", "#8a8985"
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.bbox": "tight",
        "font.size": 9,
        "axes.edgecolor": TEXT_2,
        "axes.labelcolor": TEXT,
        "axes.titlecolor": TEXT,
        "axes.titlesize": 9.5,
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": TEXT_2,
        "ytick.color": TEXT_2,
        "legend.frameon": False,
        "lines.linewidth": 1.6,
    }
)

## Carga y cobertura

Una fila por corrida esperada: cada celda dice cuantos episodios tiene esa corrida. Todas deberian tener los mismos; una celda vacia es una corrida que falta o que fallo.

In [ ]:
runs = load_runs(RUNS_ROOT, methods=METHODS_TO_SHOW)
if "comm_bytes" in runs:
    runs["comm_mb"] = runs["comm_bytes"] / 1e6

per_episode = per_episode_means(runs)
for metric in PERCENT:
    if metric in per_episode:
        per_episode[metric] *= 100

final = final_values(per_episode, FINAL_FRACTION)
summary = summarize_by_k(final)
methods = [m for m in METHODS if m in set(per_episode["method"])]
ks = sorted(per_episode["k"].unique())

coverage = per_episode.pivot_table(
    index=["method", "k"], columns="seed", values="episode", aggfunc="count"
).reindex(pd.MultiIndex.from_product([methods, EXPECTED_KS], names=["method", "k"]))
coverage = coverage.reindex(columns=EXPECTED_SEEDS)
missing = coverage.isna().to_numpy().sum()
if missing or coverage.stack().nunique() > 1:
    print(f"Atencion: faltan {missing} corridas o no todas tienen el mismo numero de episodios.")
coverage.rename(index=lambda m: METHODS.get(m, {}).get("label", m), level="method")

## Estabilidad a lo largo del entrenamiento

Recompensa por episodio (media movil de `WINDOW` episodios, promedio entre semillas), una linea por K. Greedy no aprende: sus lineas deben quedar planas. En los metodos que aprenden, esta es la curva de aprendizaje de cada K. La recompensa crece con K porque suma terminos de cada par.

In [ ]:
k_colors = dict(zip(ks, plt.cm.viridis(np.linspace(0.05, 0.85, len(ks)))))
fig, axes = plt.subplots(1, len(methods), figsize=(3.6 * len(methods) + 1.5, 3.0), squeeze=False, sharey=True)
for ax, method in zip(axes[0], methods):
    data = per_episode[per_episode["method"] == method].sort_values("episode")
    for k in ks:
        run = data[data["k"] == k]
        if run.empty:
            continue
        smooth = run.groupby("seed")["reward"].transform(lambda s: s.rolling(WINDOW, min_periods=1).mean())
        curve = smooth.groupby(run["episode"]).mean()
        ax.plot(curve.index, curve.values, color=k_colors[k], label=f"K = {k}")
    ax.set_title(METHODS[method]["label"])
    ax.set_xlabel("Episodio")
axes[0][0].set_ylabel(LABELS["reward"])
axes[0][-1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.savefig(FIG_DIR / "barrido_recompensa_por_episodio.pdf")
plt.show()

## Metricas del paper vs. K

Las cuatro metricas de desempeño de la tesis. En la tasa CUE, los puntos grises son los valores de Li et al. (2022), Fig. 3, leidos de la figura: sirven para comparar la **tendencia** (cuanto cae la tasa al pasar de 5 a 30 pares), no el valor exacto, porque el entorno esta simplificado. El delay no se compara con el paper (Fig. 8 mide otra cosa) y la recompensa esta normalizada, asi que su escala es propia.

In [ ]:
def plot_vs_k(ax, metric: str) -> None:
    """Media ± 1 desviacion entre semillas, un trazo por metodo."""
    for method in methods:
        if (metric, "mean") not in summary:
            continue
        s = summary.loc[method]
        if s[(metric, "mean")].isna().all():
            continue
        style = METHODS[method]
        ax.errorbar(
            s.index,
            s[(metric, "mean")],
            yerr=s[(metric, "std")].fillna(0.0),
            color=style["color"],
            linestyle=style["linestyle"],
            marker=style["marker"],
            markersize=4.5,
            capsize=3,
            label=style["label"],
        )
    ax.set_xticks(EXPECTED_KS)
    ax.set_xlabel("K (pares V2V)")
    ax.set_title(LABELS[metric])


fig, axes = plt.subplots(2, 2, figsize=(7.5, 5.6))
for ax, metric in zip(axes.flat, ["cue_rate_mbps", "reward", "outage_rate", "v2v_delay_ms"]):
    plot_vs_k(ax, metric)
for method in methods:
    if method in LI_FIG3_MBPS:
        ref = LI_FIG3_MBPS[method]
        axes[0][0].plot(
            list(ref), list(ref.values()), color=REFERENCE, linestyle=":", marker="x", markersize=5,
            label=f"Li et al., Fig. 3 ({METHODS[method]['label']})",
        )
axes[0][0].legend(fontsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "barrido_desempeno.pdf")
plt.show()

## Comportamiento vs. K

Que hacen los agentes con los canales y la potencia (el analogo de las Figs. 5 y 7 de Li et al.). La linea gris en *Canales en uso* es el maximo posible, min(K, 15): con mas pares que canales, compartir es inevitable. Greedy siempre transmite a potencia maxima.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8.6, 2.9))
for ax, metric in zip(axes, ["co_channel_rate", "channels_used", "max_power_rate"]):
    plot_vs_k(ax, metric)
axes[1].plot(EXPECTED_KS, [min(k, 15) for k in EXPECTED_KS], color=REFERENCE, linestyle=":", label="maximo posible")
axes[1].legend(fontsize=7.5)
axes[2].set_ylim(0, 105)
fig.tight_layout()
fig.savefig(FIG_DIR / "barrido_comportamiento.pdf")
plt.show()

## Vecinos vs. K

Con cuantos pares podria compartir modelo cada agente, a 300 m o menos. No depende del metodo (las posiciones salen de la semilla), asi que sirve para todos. Es el dato que explica el gossip: con K pequeño muchos pares quedan sin nadie con quien promediar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8))
for ax, metric in zip(axes, ["neighbors", "isolated_rate"]):
    plot_vs_k(ax, metric)
fig.tight_layout()
fig.savefig(FIG_DIR / "barrido_vecinos.pdf")
plt.show()
neighbors_table = summary.loc[methods[0], [("neighbors", "mean"), ("isolated_rate", "mean")]].round(2)
neighbors_table.columns = [LABELS["neighbors"], LABELS["isolated_rate"]]
neighbors_table

## Variabilidad entre episodios

Cada episodio sortea una geometria nueva, asi que la tasa CUE cambia de un episodio a otro aunque el metodo no cambie. La caja va del cuartil 1 al 3 de los episodios del tramo final (todas las semillas juntas) y los bigotes cubren el 5-95 %. Si las cajas de dos metodos se superponen mucho, la diferencia de medias necesita las semillas pareadas para ser creible.

In [ ]:
last_episode = per_episode.groupby(["method", "k", "seed"])["episode"].transform("max")
tail = per_episode[per_episode["episode"] > last_episode - np.ceil(FINAL_FRACTION * (last_episode + 1))]
fig, axes = plt.subplots(1, len(methods), figsize=(3.6 * len(methods) + 0.6, 3.0), squeeze=False, sharey=True)
for ax, method in zip(axes[0], methods):
    data = tail[tail["method"] == method]
    groups = [data.loc[data["k"] == k, "cue_rate_mbps"] for k in ks]
    box = ax.boxplot(groups, whis=(5, 95), showfliers=False, patch_artist=True, widths=0.55)
    for patch in box["boxes"]:
        patch.set(facecolor=METHODS[method]["color"], alpha=0.35, edgecolor=TEXT_2)
    for median in box["medians"]:
        median.set(color=TEXT)
    ax.set_xticks(range(1, len(ks) + 1), [str(k) for k in ks])
    ax.set_xlabel("K (pares V2V)")
    ax.set_title(METHODS[method]["label"])
axes[0][0].set_ylabel(LABELS["cue_rate_mbps"])
fig.savefig(FIG_DIR / "barrido_variabilidad_tasa_cue.pdf")
plt.show()

## Comunicacion, consenso y aprendizaje

Las metricas del aporte: modelos transmitidos, carga del nodo mas cargado, dispersion de pesos, perdida y episodios hasta converger. Solo existen para los metodos que entrenan o comparten modelos; para Greedy esta seccion solo lo confirma.

In [ ]:
shares = [m for m in methods if summary.loc[m, ("comm_models", "mean")].max() > 0] if ("comm_models", "mean") in summary else []
learners = [m for m in methods if ("train_loss", "mean") in summary and summary.loc[m, ("train_loss", "mean")].notna().any()]
not_applicable = [METHODS[m]["label"] for m in methods if m not in shares and m not in learners]
if not_applicable:
    print(f"{', '.join(not_applicable)}: no entrena ni comparte modelos (comunicacion 0; perdida y dispersion vacias).")

if shares or learners:
    panels = [m for m in ("comm_models", "comm_max_load", "weight_dispersion") if (m, "mean") in summary]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.0 * len(panels), 2.9))
    for ax, metric in zip(np.atleast_1d(axes), panels):
        plot_vs_k(ax, metric)
    np.atleast_1d(axes)[-1].set_yscale("log")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "barrido_comunicacion_consenso.pdf")
    plt.show()

    convergence = (
        per_episode[per_episode["method"].isin(learners)]
        .sort_values("episode")
        .groupby(["method", "k", "seed"])["reward"]
        .apply(lambda r: convergence_episode(r.to_numpy(), window=WINDOW))
    )
    display(convergence.groupby(["method", "k"]).agg(["mean", "std"]).round(1).rename(index=lambda m: METHODS.get(m, {}).get("label", m), level="method"))

## Tabla resumen

Media ± desviacion entre semillas del tramo final, por metodo y K. «—» marca una metrica que el metodo no tiene.

In [ ]:
TABLE_METRICS = [
    "cue_rate_mbps", "reward", "outage_rate", "v2v_delay_ms",
    "co_channel_rate", "channels_used", "mean_power_ratio", "max_power_rate",
    "neighbors", "isolated_rate", "comm_models", "comm_mb", "comm_max_load", "weight_dispersion", "train_loss",
]


def cell(mean: float, std: float) -> str:
    if pd.isna(mean):
        return "—"
    return f"{mean:.2f} ± {0.0 if pd.isna(std) else std:.2f}"


for method in methods:
    s = summary.loc[method]
    table = pd.DataFrame(
        {LABELS[m]: [cell(s.loc[k, (m, "mean")], s.loc[k, (m, "std")]) for k in s.index] for m in TABLE_METRICS if (m, "mean") in s},
        index=pd.Index(s.index, name="K"),
    )
    table["Semillas"] = s[("seeds", "")].to_list()
    print(METHODS[method]["label"])
    display(table)

## Tiempo de computo

Del JSON que deja cada corrida junto a su CSV. Sirve para la tabla de configuracion experimental y para estimar cuanto pedirle a Slurm en los otros metodos.

In [ ]:
meta = load_metadata(RUNS_ROOT, methods=METHODS_TO_SHOW)
if meta.empty:
    print("No hay JSON de metadatos junto a los CSV.")
else:
    minutes = meta.assign(minutos=meta["wall_time_s"] / 60).pivot_table(index=["method", "k"], columns="seed", values="minutos")
    display(minutes.round(1).rename(index=lambda m: METHODS.get(m, {}).get("label", m), level="method"))
    print("Maquinas:", ", ".join(sorted(meta["host"].dropna().unique())))

## Chequeos

Lo que tiene que cumplirse para que el barrido sea creible: la tasa CUE baja al crecer K (mas pares, mas interferencia), y la caida relativa de 5 a 30 pares se compara con la de Li et al. para el mismo metodo.

In [ ]:
for method in methods:
    rate = summary.loc[method, ("cue_rate_mbps", "mean")]
    falls = bool((rate.diff().dropna() < 0).all())
    print(f"{METHODS[method]['label']}: la tasa CUE {'baja' if falls else 'NO baja'} en cada paso de K ({', '.join(f'{v:.2f}' for v in rate)} Mbps).")
    if method in LI_FIG3_MBPS and {min(ks), max(ks)} <= set(rate.index):
        ours = rate[max(ks)] / rate[min(ks)] - 1
        ref = LI_FIG3_MBPS[method]
        theirs = ref[max(ks)] / ref[min(ks)] - 1
        print(f"  De K = {min(ks)} a K = {max(ks)}: {100 * ours:+.0f} % aqui frente a {100 * theirs:+.0f} % en Li et al.")